# Fundamentals 04 - Human Result API

Este notebook es el único `03` de fundamentals. Su trabajo es **explorar la API**, no resolver un caso OTC.

Flujo completo:

```text
normalize_output -> output_schema -> final_answer -> RunResult -> validate -> human_result
```

La regla de diseño queda visible: `result.data` conserva evidencia reusable; `result.final` es la respuesta que se entrega/renderiza; `human_result(...)` sólo imprime una vista, no decide el dominio.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass

import agentic_systems as lab
from pydantic import BaseModel

PRETTY = False

## Escenario did?ctico compartido

Todos los notebooks de `tutorials/` usan este mismo problema para que puedas comparar la API sin cambiar de caso cada vez:

```text
Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2.
Dime:
- procedimiento
- resultado final
```



In [ ]:
USER_PROMPT = """Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2.
Dime:
- procedimiento
- resultado final""".strip()

# La estructura solicitada por el usuario se materializa como datos simples.
# No es un parser ni una respuesta precocinada: sólo representa la sección `Dime:`.
REQUESTED_OUTPUTS = ["procedimiento", "resultado_final"]

lab.show({
    "prompt_usuario": USER_PROMPT,
    "salidas_solicitadas": REQUESTED_OUTPUTS,
}, title="Escenario did?ctico · visible")


## 1) `normalize_output`: todo boundary sale como diccionario

`fundamentals` debe enseñar la API base. Aquí no hay dominio: sólo formas de datos que pueden salir de tools, agents o systems.

In [ ]:
class MetricModel(BaseModel):
    name: str
    value: float


@dataclass
class MetricDataclass:
    name: str
    value: float


samples = {
    "dict": {"name": "coverage", "value": 0.99},
    "list_dict": [{"name": "a", "value": 1}, {"name": "b", "value": 2}],
    "list_scalar": ["a", "b"],
    "str": "hola",
    "number": 42,
    "none": None,
    "pydantic": MetricModel(name="quality", value=0.97),
    "dataclass": MetricDataclass(name="latency_ms", value=12.5),
}

lab.show({name: lab.normalize_output(value) for name, value in samples.items()})

## 2) `output_schema` y `final_answer`

`output_schema(...)` proyecta la respuesta pedida por el usuario. No borra la evidencia completa; sólo construye el entregable final.

In [ ]:
raw_payload = {
    "operation": "divide",
    "result": 7.0,
    "debug": {"a": 21, "b": 3},
    "rows": [{"operation": "divide", "result": 7.0, "extra": "evidence"}],
}

single_schema = lab.output_schema(fields=["operation", "result"])
rows_schema = lab.output_schema(fields=["operation", "result"], many=True)

lab.show({
    "single_schema": single_schema.model_dump(mode="json"),
    "single_final": lab.final_answer(raw_payload, schema=single_schema),
    "rows_final": lab.final_answer(raw_payload, schema=rows_schema),
    "fallback_text_final": lab.final_answer(text="Respuesta libre controlada"),
})

## 3) Tool mínima para generar un `RunResult`

La tool es deliberadamente simple. Sirve para producir un `RunResult` real sin depender de Bedrock, Athena ni casos de negocio.

In [ ]:
@lab.tool
def divide_numbers(a: float, b: float) -> dict:
    """Divide dos números y devuelve un dict auditable."""
    if b == 0:
        return {"ok": False, "operation": "divide", "error": "division_by_zero"}
    return {"ok": True, "operation": "divide", "result": a / b}


raw_result = divide_numbers.run({"a": 21, "b": 3})
lab.show({"tool_data": raw_result.data, "tool_event_count": len(raw_result.tool_events)})

## 4) Contrato + policy declarativos

`AgentContract` define qué debe cumplirse. `RunPolicy` define límites de ejecución. `ContractPolicySpec` empaqueta ambos para que puedan inspeccionarse, reutilizarse y validarse antes de gastar runtime/tokens.

In [ ]:
division_spec = lab.ContractPolicySpec(
    name="fundamentals.divide_once",
    description="Debe llamar una vez a divide_numbers y producir una salida exitosa.",
    contract=lab.AgentContract(
        must_call=["divide_numbers"],
        tool_expectation=lab.expect.exactly("divide_numbers"),
        completion="when_required_tools_satisfied",
        failure_policy="no_unresolved",
        expected_output={"operation": "divide", "result": 7.0},
        expected_tool_outputs={"divide_numbers": {"ok": True, "operation": "divide"}},
    ),
    policy=lab.RunPolicy(
        max_turns=3,
        max_tool_calls=1,
        temperature=0.0,
        tool_choice="divide_numbers",
        finalize="after_required_tools",
        trace="compact",
        strict=True,
    ),
    tags=["fundamentals", "result", "policy"],
)

lab.show({
    "spec": division_spec.describe(),
    "static_check": division_spec.check(available_tools=["divide_numbers"]).to_dict(),
    "agent_kwargs_keys": sorted(division_spec.agent_kwargs()),
})

## 5) `RunResult`: envelope estable

El envelope guarda respuesta, evidencia, tool events, usage, runtime y validación. Esta estructura es la que luego comparten tools, agents, systems, integrations y notebooks.

In [ ]:
result: lab.RunResult = raw_result
result.final = lab.final_answer(result.data, schema=single_schema, text=result.text)
validation = result.validate(division_spec.contract)
result.validation = validation.to_dict()
result.ok = result.ok and validation.ok

lab.show({
    "ok": result.ok,
    "validation": result.validation,
    "data_evidence": result.data,
    "final_answer": result.final,
})

## 6) `normalized()` y `trace(...)`

`normalized()` es el contrato público para comparar salidas entre engines/frameworks. `trace(...)` conserva más detalle operativo sin cambiar la respuesta final.

In [ ]:
lab.show({
    "normalized": result.normalized(),
    "compact_trace": result.trace("compact"),
})

## 7) `human_result`: render declarativo

La vista humana lee `RunResult`. No debe inventar SQL, tablas ni bloques de dominio; eso debe venir declarado en `result.final`/`result.data`/`tool_events`.

In [ ]:
lab.human_result(
    result,
    title="Human result · RunResult + final answer + contrato",
    expected_tools=division_spec.contract.tool_expectation,
    pretty=PRETTY,
)

## 8) Fallo temprano de contrato/policy

Esto enseña por qué conviene validar antes de ejecutar: si el contrato pide dos tools pero la policy permite una sola llamada, el problema se detecta sin gastar runtime.

In [ ]:
bad_spec = lab.ContractPolicySpec(
    name="fundamentals.bad_policy",
    contract=lab.AgentContract(must_call=["divide_numbers", "missing_tool"]),
    policy=lab.RunPolicy(max_tool_calls=1),
)

lab.show(bad_spec.check(available_tools=["divide_numbers"]).to_dict())

## Final answer del escenario did?ctico compartido

La misma pregunta se proyecta a un entregable de usuario: procedimiento + resultado final. La evidencia puede ser más grande, pero `final_answer(...)` controla lo que se entrega.

In [ ]:
def user_problem_payload() -> dict:
    """Payload did?ctico del escenario did?ctico para cerrar con final_answer(...)."""
    return {
        "procedimiento": [
            "10 + 20 = 30",
            "30 - 9 = 21",
            "21 * 4 = 84",
            "84 / 2 = 42",
        ],
        "resultado_final": 42,
        "ok": True,
    }


default_payload = user_problem_payload()
user_schema = lab.output_schema(fields=["procedimiento", "resultado_final", "ok"])

lab.show({
    "raw_payload": default_payload,
    "final_answer": lab.final_answer(default_payload, schema=user_schema),
})

## Lo importante

- `fundamentals` explora la API, no el dominio OTC.
- `normalize_output(...)` vuelve predecibles las salidas de boundary.
- `final_answer(...)` materializa el entregable final.
- `RunResult` conserva evidencia, runtime, tool events y validación.
- `ContractPolicySpec.check(...)` valida antes de ejecutar.
- `RunResult.validate(...)` valida lo que realmente pasó.
- `human_result(...)` renderiza una estructura declarada; no decide reglas de negocio.

## Coverage API de este notebook

Esta tabla deja explícito qué parte de Agentic Systems queda materializada aquí.

In [ ]:
api_coverage = [
    {
        "api": "normalize_output",
        "description": "Normaliza boundaries para que toda salida salga como dict."
    },
    {
        "api": "output_schema",
        "description": "Declara la forma de la salida final con un schema estable."
    },
    {
        "api": "final_answer",
        "description": "Proyecta un resumen humano desde data + texto + schema."
    },
    {
        "api": "RunResult",
        "description": "Agrupa texto, data, metadata y validacion en un envelope unico."
    },
    {
        "api": "validate",
        "description": "Comprueba el contrato sin mezclarlo con presentacion."
    },
    {
        "api": "human_result",
        "description": "Renderiza la salida humana sin perder la evidencia interna."
    },
    {
        "api": "default final answer projection",
        "description": "Muestra la proyeccion final del problema comun del tutorial."
    }
]

lab.show({'notebook': '04_human_result_api.ipynb', 'api_coverage': api_coverage})

## S?mbolos API explicados

Este notebook se alinea con `docs/API.md` y ense?a estos s?mbolos p?blicos:

- `RunResult`: Envelope estable de ejecuci?n.
- `normalize_output`: Normalizaci?n de boundaries a diccionario.
- `OutputSchema / output_schema`: Schema p?blico de salida.
- `FINAL_ANSWER_SCHEMA_VERSION / final_answer`: Formato p?blico de respuesta final.
- `AgenticOutput / RuntimeInfo / UsageInfo`: Tipos p?blicos del resultado normalizado.
- `OutputToolEvent / OutputValidation / TraceEvent`: Tipos p?blicos para acciones, validaci?n y traza.
- `human_result / human_results / print_human_result / print_human_results`: Render humano p?blico.
- `run_result_output / run_result_view / run_result_summary / tool_result_summary`: Vistas p?blicas de resultado.

- `AGENT_OUTPUT_SCHEMA_VERSION / OUTPUT_SCHEMA_VERSION / AGENTIC_OUTPUT_SCHEMA_VERSION`: Versiones p?blicas de schemas de salida.
- `agent_output / agent_output_mapper / make_agent_output_mapper`: Mappers p?blicos para convertir salidas de agente a envelopes estables.
- `compare`: Helper p?blico para comparar estructuras/salidas en notebooks.

